In [4]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 99.4 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-

In [2]:
import os

yaml_path = "/kaggle/input/datasets/singertv/fabric-defect-v2-checkpoint/dataset_oversampled.yaml"

with open(yaml_path) as f:
    content = f.read()
print(content)

for line in content.splitlines():
    if line.startswith(("train:", "val:", "test:")):
        path = line.split(":", 1)[1].strip()
        print(path, "→ EXISTS" if os.path.exists(path) else "→ MISSING")

# Auto-generated by oversample_seam.py
# Train points to the oversampled set; val/test stay on the original, untouched data.
train: /kaggle/working/oversampled/images/train
val: /kaggle/input/datasets/singertv/fabric-processed/processed/images/val
test: /kaggle/input/datasets/singertv/fabric-processed/processed/images/test

nc: 5
names: ['Stain', 'Thread', 'Warp_Weft', 'hole', 'seam']

/kaggle/working/oversampled/images/train → MISSING
/kaggle/input/datasets/singertv/fabric-processed/processed/images/val → EXISTS
/kaggle/input/datasets/singertv/fabric-processed/processed/images/test → EXISTS


In [5]:
from ultralytics import YOLO

best_model = YOLO("/kaggle/input/datasets/singertv/fabric-defect-v2-checkpoint/best.pt")

metrics = best_model.val(
    data="/kaggle/input/datasets/singertv/fabric-defect-v2-checkpoint/dataset_oversampled.yaml",
    split="test"
)

print("📊 Test Set Results (v2 — seam oversampled):")
print(f"mAP50      : {metrics.box.map50:.4f}")
print(f"mAP50-95   : {metrics.box.map:.4f}")
print(f"Precision  : {metrics.box.mp:.4f}")
print(f"Recall     : {metrics.box.mr:.4f}")

print("\n📊 Per-Class Results:")
for i, name in enumerate(metrics.names.values()):
    print(f"  {name:12s} — AP50: {metrics.box.ap50[i]:.4f}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.71 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 93 layers, 25,842,655 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 34.1±12.1 MB/s, size: 117.8 KB)
val: Scanning /kaggle/input/datasets/singertv/fabric-processed/processed/labels/test... 329 images, 161 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 329/329 273.4it/s 1.2s0.0s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/singertv/fabric-processed/processed/labels is not writable, cache not saved.
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 4, len(boxes) = 253. To resolve this only boxes will be used a

In [6]:
import shutil

# Zip the val results folder
shutil.make_archive('/kaggle/working/val_results_v2', 'zip', '/kaggle/working/runs/detect/val')
print("Zipped val results ✅")

# If you still have the full training run output in this session too:
import os
if os.path.exists('/kaggle/working/fabric-defect'):
    shutil.make_archive('/kaggle/working/training_run_v2', 'zip', '/kaggle/working/fabric-defect')
    print("Zipped training run ✅")

Zipped val results ✅
